# Import Librairie

In [1]:
import pandas as pd
import openpyxl

# Données Générales

In [2]:
communes = pd.read_csv("../data/raw/communes-france-2025.csv", sep=";")
# dans la colonne "code_insee" ajoute un zéro devant les codes insee qui en ont que 4 chiffres
communes["code_insee"] = communes["code_insee"].apply(lambda x: "0" + str(x) if len(str(x)) == 4 else str(x))
# filtrer les communes pour ne garder que celles qui ont une population de plus de 1000 habitants
communes_a_garder = communes[communes["population"] > 20000]


C:\Users\alice\AppData\Local\Temp\ipykernel_31460\2828831195.py:1: DtypeWarning: Columns (0: code_insee, 1: dep_code, 2: canton_code, 3: epci_code, 4: code_insee_centre_zone_emploi, 5: code_unite_urbaine) have mixed types. Specify dtype option on import or set low_memory=False.
  communes = pd.read_csv("../data/raw/communes-france-2025.csv", sep=";")


In [3]:
lst_villes = communes_a_garder["nom_standard"].tolist()
len(lst_villes)

483

# Web Scrapping Ville de Reve

In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--remote-debugging-port=9222")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

data_list = []   # <-- IMPORTANT : en dehors de la boucle
lst_villes = communes_a_garder["nom_standard"].tolist()

for ville in lst_villes:

    url = f"https://villedereve.fr/ville/{communes_a_garder[communes_a_garder['nom_standard'] == ville]['code_insee'].values[0]}-{ville.lower()}"
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.get(url)
    time.sleep(1)  # petite pause pour laisser charger

    # -------------------------
    # RÉCUPÉRATION DU SCORE
    # -------------------------
    try:
        score_element = driver.find_element(By.CSS_SELECTOR, "#city-score h3")
        score = score_element.text.strip() + "100"
    except:
        score = "NA"

    print("Score :", score)

    # -------------------------
    # RÉCUPÉRATION DES DESCRIPTIONS
    # -------------------------
    try:
        description_elements = driver.find_elements(By.CSS_SELECTOR, "p.description")
        descriptions = [desc.text.strip() for desc in description_elements]

        if len(descriptions) == 0:
            descriptions = "NA"
    except:
        descriptions = "NA"

    print("Descriptions :", descriptions)

    # -------------------------
    # STOCKAGE
    # -------------------------
    ville_data = {
        "ville": ville,
        "score": score,
        "presentation": descriptions
    }

    data_list.append(ville_data)

    driver.quit()

    # -------------------------
    # DATAFRAME FINAL
    # -------------------------
    df_villes = pd.DataFrame(data_list)

    df_final = pd.merge(
        communes_a_garder,
        df_villes,
        left_on="nom_standard",
        right_on="ville",
        how="left"
    )

    df_final.to_excel("../data/processed/donnees_generale_filtrer.xlsx", index=False)

Score : 71.3/100
Descriptions : ["Bourg-en-Bresse est une commune du centre-est de la France, dans le département de l'Ain et la région d'Auvergne-Rhône-Alpes. Il s'agit d'une petite ville de 42 370 habitants à 58 km au nord-est de Lyon. La commune est au coeur d'une agglomération de 63 000 habitants.", 'En 2026, Bourg-en-Bresse a un score général de 71.3 / 100. Elle est 204ᵉ / 1216 du palmarès des petites villes en 2026.', "Il s'agit de la 1ᵉ ville la plus peuplée de l'Ain et 3ᵉ ville la plus dense de l'Ain.", 'La superficie de Bourg-en-Bresse est de 2400 hectares (24 km²). Bourg-en-Bresse connaît un climat océanique dégradé des plaines du Centre et du Nord et sont altitude moyenne est de 239 mètres (entre 220m et 273m).', "Très dense (1766 habitants au km²), la commune est extrêmement urbanisée (avec 63.5% de sa surface qui est artificialisée, dont 0% d'espaces verts urbains). Les zones agricoles représentent 21.2% de la surface communale et les espaces naturels représentent 15.3% de

In [ ]:
# garder les coordonnées des villes dans une liste avec le nom de la ville, la latitude et la longitude
coordonnees_villes = []
for index, row in communes_a_garder.iterrows():
    ville = row["nom_standard"]
    latitude = row["latitude_mairie"]
    longitude = row["longitude_mairie"]
    coordonnees_villes.append({
        "ville": ville,
        "latitude": latitude,
        "longitude": longitude
    })
coordonnees_df = pd.DataFrame(coordonnees_villes)
coordonnees_df
coordonnees_df.to_excel("../data/processed/coordonnees_villes.xlsx", index=False)
# read le exel
coordonnees_df = pd.read_excel("../data/processed/coordonnees_villes.xlsx")

# Education

In [17]:
education = pd.read_csv("../data/raw/annuaire-de-l-education.csv", sep=";")
education = education[["Identifiant_de_l_etablissement", "Nom_etablissement", "Type_etablissement", "Statut_public_prive","Adresse_1","Code_postal","Code_commune","Nom_commune","Code_departement","Ecole_maternelle",	"Ecole_elementaire"	,"Voie_generale"	,"Voie_technologique",	"Voie_professionnelle",	"Telephone","Web","Mail",	"Restauration",	"Hebergement",	"ULIS",	"Apprentissage",	"Segpa"	,"Section_arts"	,"Section_cinema",	"Section_theatre",	"Section_sport",	"Section_internationale",	"Section_europeenne",	"Lycee_Agricole",	"Lycee_militaire",	"Lycee_des_metiers",	"Post_BAC","position",	"Libelle_departement",	"Libelle_academie",	"Libelle_region","date_ouverture"]]

education_a_garder = education[education["Nom_commune"].isin(lst_villes)]

education_a_garder.head()
education_a_garder.to_excel("../data/processed/education_filtrer.xlsx", index=False)

C:\Users\alice\AppData\Local\Temp\ipykernel_32428\2876586733.py:1: DtypeWarning: Columns (0: rpi_disperse, 1: Code_type_contrat_prive) have mixed types. Specify dtype option on import or set low_memory=False.
  education = pd.read_csv("../data/raw/annuaire-de-l-education.csv", sep=";")


# Logements

In [18]:
logement = pd.read_csv("../data/raw/stats_whole_period.csv", sep=",")
logement_a_garder = logement[logement["libelle_geo"].isin(lst_villes)]
logement_a_garder.head()
logement_a_garder.to_excel("../data/processed/logement_filtrer.xlsx", index=False)

# Sport

In [19]:
sport = pd.read_csv("../data/raw/equipements-sportifs.csv", sep=";")
sport_a_garder = sport[sport["Commune nom"].isin(lst_villes)]
sport_a_garder.head()
sport_a_garder.to_excel("../data/processed/sport_filtrer.xlsx", index=False)

C:\Users\alice\AppData\Local\Temp\ipykernel_32428\1175669173.py:1: DtypeWarning: Columns (0: Code Postal, 1: Commune INSEE, 2: Département Code, 3: Bassin de vie Code, 4: Arrondissement Code, 5: Département Code Complet) have mixed types. Specify dtype option on import or set low_memory=False.
  sport = pd.read_csv("../data/raw/equipements-sportifs.csv", sep=";")


# Culture

In [20]:
culture = pd.read_csv("../data/raw/base-des-lieux-et-des-equipements-culturels.csv", sep=";")
culture_a_garder = culture[culture["libelle_geographique"].isin(lst_villes)]
culture_a_garder.head()
culture_a_garder.to_excel("../data/processed/culture_filtrer.xlsx", index=False)

C:\Users\alice\AppData\Local\Temp\ipykernel_32428\3959625545.py:1: DtypeWarning: Columns (0: Code Postal, 1: Archéologie détail, 2: Fonction_3, 3: Fonction_4, 4: Multiplexe, 5: Jauge_du_theatre, 6: Nom_de_l'Illustre, 7: Annee_Label_Appellation) have mixed types. Specify dtype option on import or set low_memory=False.
  culture = pd.read_csv("../data/raw/base-des-lieux-et-des-equipements-culturels.csv", sep=";")


# Tourisme (a revoir)

In [19]:
import requests
import time

API_KEY = "4c775015-0f12-4366-bc70-d8dbe0404ec0"
BASE_URL = "https://api.datatourisme.fr/v1/catalog"

headers = {
    "X-API-Key": API_KEY
}

params = {
    "filters": "isLocatedAt.address.hasAddressCity.insee=75056",
    "page_size": 100
}

all_poi = []
url = BASE_URL

while url:
    r = requests.get(url, headers=headers, params=params)
    data = r.json()

    all_poi.extend(data["objects"])
    url = data["meta"]["next"]   # pagination officielle ✅
    params = None                # IMPORTANT : next contient déjà les params

    time.sleep(0.1)              # respect quotas

print(f"{len(all_poi)} POI récupérés")

KeyError: 'next'